In [ ]:
cd /kaggle/working/

In [ ]:
import shutil
shutil.rmtree("/kaggle/working/BaNLAD_SwinDocSegmenter")

In [ ]:
!git clone https://github.com/RadeenXALNW/BaNLAD_SwinDocSegmenter.git

In [1]:
cd /kaggle/working/BaNLAD_SwinDocSegmenter

/kaggle/working/BaNLAD_SwinDocSegmenter


In [ ]:

!pip install 'git+https://github.com/facebookresearch/detectron2.git'


In [ ]:
from detectron2 import _C

In [2]:
import detectron2.utils.comm as comm
from detectron2.checkpoint import DetectionCheckpointer
from detectron2.config import get_cfg
from detectron2.data import MetadataCatalog, build_detection_train_loader
from detectron2.modeling import GeneralizedRCNNWithTTA
from detectron2.evaluation import (
    CityscapesInstanceEvaluator,
    CityscapesSemSegEvaluator,
    COCOEvaluator,
    COCOPanopticEvaluator,
    DatasetEvaluators,
    LVISEvaluator,
    SemSegEvaluator,
    verify_results,
)
from detectron2.projects.deeplab import add_deeplab_config, build_lr_scheduler
from detectron2.solver.build import maybe_add_gradient_clipping
from detectron2.utils.logger import setup_logger

from detectron2.data.datasets import register_coco_instances
from detectron2.data.datasets.coco import convert_to_coco_json
from detectron2.engine import (
    DefaultTrainer,
    default_argument_parser,
    default_setup,
    hooks,
    launch,
    create_ddp_model,
    AMPTrainer,
    SimpleTrainer
)

from IPython.display import FileLink
import sys
# torch
import torch

import gc

from detectron2.utils.memory import retry_if_cuda_oom
from detectron2.utils.logger import setup_logger
from detectron2.checkpoint import DetectionCheckpointer
from detectron2.modeling import build_model
from detectron2.evaluation import COCOEvaluator, inference_on_dataset
import detectron2.data.transforms as T
from detectron2.data import detection_utils as utils
from detectron2.data import DatasetCatalog, MetadataCatalog, build_detection_test_loader, build_detection_train_loader, DatasetMapper
from detectron2.utils.visualizer import Visualizer
from detectron2.structures import BoxMode
from detectron2.engine import DefaultPredictor, DefaultTrainer
from detectron2.config import get_cfg
from detectron2 import model_zoo

import pandas as pd
import numpy as np
from tqdm.notebook import tqdm  # progress bar
import matplotlib.pyplot as plt
import json
import cv2
import copy
from typing import Optional


from IPython.display import FileLink
import sys
# torch
import torch

import gc

import warnings
# Ignore "future" warnings and Data-Frame-Slicing warnings.
warnings.filterwarnings('ignore')

setup_logger()

<_Logger detectron2 (DEBUG)>

In [ ]:
!pip install -r requirements.txt

In [ ]:
!cd maskdino/modeling/pixel_decoder/ops && sh make.sh


In [ ]:
# !pip install gdown 
# !gdown 17F-81EFkyKXhOIt7m65rYfaKeQTu7rL1
# !gdown 1180cKcF1gRwI6RMI926sRsoiugxWWQty
# !gdown 1vjwERzB-e7OL6cSg0gAYxRHOJR9j1PAg
!gdown 1LtlZSWUulPptyoklz8rsXwWa0FcVC8K8

In [ ]:
import os 
os.makedirs("output")

In [ ]:
!mv /kaggle/working/BaNLAD_SwinDocSegmenter/model_final.pth /kaggle/working/BaNLAD_SwinDocSegmenter/output

In [ ]:
!mv /kaggle/working/BaNLAD_SwinDocSegmenter/last_checkpoint /kaggle/working/BaNLAD_SwinDocSegmenter/output

In [3]:
from maskdino import (
    COCOInstanceNewBaselineDatasetMapper,
    COCOPanopticNewBaselineDatasetMapper,
    InstanceSegEvaluator,
    MaskFormerSemanticDatasetMapper,
    SemanticSegmentorWithTTA,
    add_maskformer2_config,
    DetrDatasetMapper,
)

In [ ]:
cfg = get_cfg()
# for poly lr schedule
add_deeplab_config(cfg)
add_maskformer2_config(cfg)
cfg.merge_from_file("/kaggle/working/BaNLAD_SwinDocSegmenter/maskdino_R50_bs16_50ep_3s.yaml")
print(cfg)

In [ ]:
from train_net import Trainer

In [4]:
### Data load
torch.cuda.empty_cache()
from pathlib import Path

TRAIN_IMG_DIR = Path("/kaggle/input/banlad2599-bangla-newspaper-layout-dataset/All_Crumpled_Images")

TRAIN_COCO_PATH=Path("/kaggle/working/BaNLAD_SwinDocSegmenter/updated_coco.json")

# Training output directory
OUTPUT_DIR = Path("output")
OUTPUT_MODEL = OUTPUT_DIR/"model_final.pth"

# Path to your pretrained model weights
PRETRAINED_PATH = Path("/kaggle/working/model_final.pth")

In [5]:
import json
### Coco Annotation
from pycocotools.coco import COCO

with TRAIN_COCO_PATH.open() as f:
    train_dict = json.load(f)

train_coco_labels=COCO(annotation_file=TRAIN_COCO_PATH)

print("#### LABELS AND METADATA LOADED ####")

loading annotations into memory...
Done (t=6.95s)
creating index...
index created!
#### LABELS AND METADATA LOADED ####


In [6]:
#Decisions

from datetime import datetime

# if False, model is set to `PRETRAINED_PATH` model
is_train = True

# if True, evaluate on validation dataset
is_evaluate = False

# if True, run inference on test dataset
is_inference = True

# if True and `is_train` == True, `PRETRAINED_PATH` model is trained further
is_resume_training = True

# Perform augmentation
is_augment = True

SEED = 1234

# Model path based on Decisions
MODEL_PATH = OUTPUT_MODEL if is_train else PRETRAINED_PATH

In [7]:
print("There are " + str(len(train_dict['categories'])) + " categories.\n")
# print("There are " + str(len(test_dict['images']) + len(train_dict['images'])) + " images in the dataset.")
print("There are " + str(len(train_dict['images'])) + " images in the train set.")
# print("There are " + str(len(test_dict['images'])) + " images in the test set.\n")
print("There are " + str(len(train_dict['annotations'])) + " annotations in the train set.\n")

print("We will focus on mainly categories, images and annotations.")

There are 6 categories.

There are 2726 images in the train set.
There are 641353 annotations in the train set.

We will focus on mainly categories, images and annotations.


In [8]:
def organize_coco_data(data_dict: dict) -> tuple[list[str], list[dict], list[dict]]:
    thing_classes: list[str] = []

    # Map Category Names to IDs
    for cat in data_dict['categories']:
        thing_classes.append(cat['name'])

    # Images
    images_metadata: list[dict] = data_dict['images']

    # Convert COCO annotations to detectron2 annotations format
    data_annotations = []
    for ann in data_dict['annotations']:
        # coco format -> detectron2 format
        annot_obj = {
            # Annotation ID
            "id": ann['id'],

            # Segmentation Polygon (x, y) coordinnates
            "gt_masks": ann['segmentation'],

            # Image ID for this annotation (Which image does this annotation belong to?)
            "image_id": ann['image_id'],

            # Category Label (0: paragraph, 1: text box, 2: image, 3: table)
            "category_id": ann['category_id'],

            "x_min": ann['bbox'][0],  # left
            "y_min": ann['bbox'][1],  # top
            "x_max": ann['bbox'][0] + ann['bbox'][2],  # left+width
            "y_max": ann['bbox'][1] + ann['bbox'][3]  # top+height
        }
        data_annotations.append(annot_obj)

    return thing_classes, images_metadata, data_annotations

In [9]:
thing_classes, images_metadata, data_annotations = organize_coco_data(train_dict)

# thing_classes_test, images_metadata_test, _ = organize_coco_data(test_dict)

print(thing_classes)
# thing_classes = [cls for cls in thing_classes if cls != 'newspaper']

print(len(thing_classes))

['news', 'image', 'news', 'paragraph', 'table', 'text-box']
6


In [10]:
train_metadata = pd.DataFrame(images_metadata)
train_metadata = train_metadata[['id', 'file_name', 'width', 'height']]
train_metadata = train_metadata.rename(columns={"id": "image_id"})
print("train_metadata size=", len(train_metadata))
train_metadata.head(5)

train_metadata size= 2726


,image_id,file_name,width,height
0,0,image_249_png.rf.24e6802439adcea910092682ce070...,2592,4608
1,1,image_232_png.rf.206a044d134b5893141ae01bf4a42...,2592,4608
2,2,image_202_png.rf.14185520c66db852b1c1a0f12b9aa...,3000,4000
3,3,image_198_png.rf.10d4c751aa2909875cf306d0eb514...,2592,4608
4,4,image_165_png.rf.054e36e83984ebbe210819bbbed0a...,3024,4032


In [11]:
train_annot_df = pd.DataFrame(data_annotations)
print("train_annot_df size=", len(train_annot_df))
train_annot_df.head(5)

train_annot_df size= 641353


,id,gt_masks,image_id,category_id,x_min,y_min,x_max,y_max
0,0,"[[1377.5, 341.255, 1612.5, 341.255, 1612.5, 44...",0,5,1378,341,1613.000,447.25
1,1,"[[1616, 490, 529, 510, 525, 904, 528, 1366, 51...",0,2,515,490,1680.000,2206.00
2,2,"[[2107.5, 427.5, 1891.25, 431.25, 1630, 457.5,...",0,3,1618,324,2340.525,457.75
3,3,"[[1351.275, 323.75, 1362.525, 457.5, 1632.525,...",0,2,1351,309,2349.750,472.75
4,4,"[[1618.775, 487.5, 1782.525, 467.5, 1970.025, ...",0,5,1619,468,1981.500,596.75


In [12]:
#Formatting Data for detectron2
def convert_coco_to_detectron2_format(
    imgdir: Path,
    metadata_df: pd.DataFrame,
    annot_df: Optional[pd.DataFrame] = None,
    target_indices: Optional[np.ndarray] = None,
):

    dataset_dicts = []
    for _, train_meta_row in tqdm(metadata_df.iterrows(), total=len(metadata_df)):
        # Iterate over each image
        image_id, filename, width, height = train_meta_row.values

        annotations = []
        
        # If train/validation data, then there will be annotations
        if annot_df is not None:
            for _, ann in annot_df.query("image_id == @image_id").iterrows():
                # Get annotations of current iteration's image
                class_id = ann["category_id"]
                gt_masks = ann["gt_masks"]
                bbox_resized = [
                    float(ann["x_min"]),
                    float(ann["y_min"]),
                    float(ann["x_max"]),
                    float(ann["y_max"]),
                ]

                annotation = {
                    "bbox": bbox_resized,
                    "bbox_mode": BoxMode.XYXY_ABS,
                    "segmentation": gt_masks,
                    "category_id": class_id,
                }

                annotations.append(annotation)
        # coco format -> detectron2 format dict
        record = {
            "file_name": str(imgdir/filename),
            "image_id": image_id,
            "width": width,
            "height": height,
            "annotations": annotations
        }

        dataset_dicts.append(record)

    if target_indices is not None:
        dataset_dicts = [dataset_dicts[i] for i in target_indices]

    return dataset_dicts

In [13]:
DatasetCatalog.clear()


In [14]:
TRAIN_SPLIT = 0.90
n_dataset = len(train_metadata)
n_train = int(n_dataset * TRAIN_SPLIT)
print("n_dataset", n_dataset, "n_train", n_train, "n_val", n_dataset-n_train)

np.random.seed(SEED)

inds = np.random.permutation(n_dataset)
train_inds, valid_inds = inds[:n_train], inds[n_train:]

n_dataset 2726 n_train 2453 n_val 273


In [15]:
#Registering and Loading Data for detectron2
DATA_REGISTER_TRAINING = "pub_train"
DATA_REGISTER_VALID    = "pub_val"
DATA_REGISTER_TEST     = "badlad_test"

In [16]:
# Register Training data
if is_train:
    DatasetCatalog.register(
        DATA_REGISTER_TRAINING,
        lambda: convert_coco_to_detectron2_format(
            TRAIN_IMG_DIR,
            train_metadata,
            train_annot_df,
            target_indices=train_inds,
        ),
    )

    # Set Training data categories
    MetadataCatalog.get(DATA_REGISTER_TRAINING).set(thing_classes=thing_classes)

    dataset_dicts_train = DatasetCatalog.get(DATA_REGISTER_TRAINING)
    metadata_dicts_train = MetadataCatalog.get(DATA_REGISTER_TRAINING)

    print("dicts training size=", len(dataset_dicts_train))
    print("################")

  0%|          | 0/2726 [00:00<?, ?it/s]

dicts training size= 2453
################


In [ ]:
def custom_mapper(dataset_dict):
    dataset_dict = copy.deepcopy(dataset_dict)
    image = utils.read_image(dataset_dict["file_name"], format="BGR")

    transform_list = [#T.RandomBrightness(0.8, 1.2),
#                       T.RandomFlip(prob=0.5, horizontal=False, vertical=True)
                        T.Resize((512,512))
                      #T.RandomFlip(prob=0.5, horizontal=True, vertical=False)
                      ]
    image, transforms = T.apply_transform_gens(transform_list, image)

    dataset_dict["image"] = torch.as_tensor(
        image.transpose(2, 0, 1).astype("float32"))

    annos = [
        utils.transform_instance_annotations(obj, transforms, image.shape[:2])
        for obj in dataset_dict.pop("annotations")
        if obj.get("iscrowd", 0) == 0
    ]
    instances = utils.annotations_to_instances(annos, image.shape[:2])

    dataset_dict["instances"] = utils.filter_empty_instances(instances)


In [ ]:
from train_net import Trainer
class AugTrainer1(Trainer):
    def build_train_loader(cls, cfg):
        return build_detection_train_loader(cfg, mapper=custom_mapper)

In [ ]:
!gdown 1DCxG2MCza_z-yB3bLcaVvVR4Jik00Ecq

In [ ]:
import warnings

# Suppress all warnings
warnings.filterwarnings("ignore")


In [ ]:
from train_net import Trainer
if is_train:  
    cfg = get_cfg()
    # for poly lr schedule
    add_deeplab_config(cfg)
    add_maskformer2_config(cfg)
    cfg.DATASETS.TRAIN = (DATA_REGISTER_TRAINING,)
    cfg.merge_from_file("maskdino_R50_bs16_50ep_3s.yaml")
    cfg.MODEL.DEVICE = "cuda" 
    cfg.SOLVER.AMP.ENABLED = True
    cfg.DATALOADER.NUM_WORKERS =2
    
    cfg.MODEL.WEIGHTS = "/kaggle/working/BaNLAD_SwinDocSegmenter/output/model_final.pth"
    # cfg.INPUT.CROP.ENABLED=True
    cfg.SOLVER.AMP.ENABLED = True
    cfg.SOLVER.IMS_PER_BATCH = 1
    cfg.SOLVER.BASE_LR = 0.001
    cfg.SOLVER.STEPS = (500, 1000)
    cfg.SOLVER.WARMUP_ITERS = 5
    cfg.INPUT.DATASET_MAPPER_NAME = "coco_instance_lsj"
    cfg.INPUT.IMAGE_SIZE=512
    
    # Maximum number of iterations
    cfg.SOLVER.MAX_ITER = 60000
    cfg.TEST.EVAL_PERIOD=80000
    cfg.SOLVER.CHECKPOINT_PERIOD = 5000

    # cfg.SOLVER.STEPS = (500, 1000) # must be less than MAX_ITER

    cfg.SOLVER.GAMMA = 0.09
    # Small value == Frequent save need a lot of storage.
    cfg.MODEL.ROI_HEADS.BATCH_SIZE_PER_IMAGE = 8
    cfg.MODEL.SEM_SEG_HEAD.NUM_CLASSES = 6
    cfg.OUTPUT_DIR = str(OUTPUT_DIR)
    trainer = Trainer(cfg)
    print(trainer)
    trainer.resume_or_load(resume=is_resume_training)

    trainer.train()

criterion.weight_dict  {'loss_ce': 4.0, 'loss_mask': 5.0, 'loss_dice': 5.0, 'loss_bbox': 5.0, 'loss_giou': 2.0, 'loss_ce_interm': 4.0, 'loss_mask_interm': 5.0, 'loss_dice_interm': 5.0, 'loss_bbox_interm': 5.0, 'loss_giou_interm': 2.0, 'loss_ce_dn': 4.0, 'loss_mask_dn': 5.0, 'loss_dice_dn': 5.0, 'loss_bbox_dn': 5.0, 'loss_giou_dn': 2.0, 'loss_ce_interm_dn': 4.0, 'loss_mask_interm_dn': 5.0, 'loss_dice_interm_dn': 5.0, 'loss_bbox_interm_dn': 5.0, 'loss_giou_interm_dn': 2.0, 'loss_ce_0': 4.0, 'loss_mask_0': 5.0, 'loss_dice_0': 5.0, 'loss_bbox_0': 5.0, 'loss_giou_0': 2.0, 'loss_ce_interm_0': 4.0, 'loss_mask_interm_0': 5.0, 'loss_dice_interm_0': 5.0, 'loss_bbox_interm_0': 5.0, 'loss_giou_interm_0': 2.0, 'loss_ce_dn_0': 4.0, 'loss_mask_dn_0': 5.0, 'loss_dice_dn_0': 5.0, 'loss_bbox_dn_0': 5.0, 'loss_giou_dn_0': 2.0, 'loss_ce_interm_dn_0': 4.0, 'loss_mask_interm_dn_0': 5.0, 'loss_dice_interm_dn_0': 5.0, 'loss_bbox_interm_dn_0': 5.0, 'loss_giou_interm_dn_0': 2.0, 'loss_ce_1': 4.0, 'loss_mask_1':

  0%|          | 0/2726 [00:00<?, ?it/s]

[08/19 22:12:56 d2.data.build]: Distribution of instances among all 6 categories:
|  category  | #instances   |  category  | #instances   |  category  | #instances   |
|:----------:|:-------------|:----------:|:-------------|:----------:|:-------------|
|    news    | 0            |   image    | 22992        |    news    | 44399        |
| paragraph  | 177332       |   table    | 1822         |  text-box  | 324198       |
|            |              |            |              |            |              |
|   total    | 570743       |            |              |            |              |
[08/19 22:12:56 d2.data.build]: Using training sampler TrainingSampler
[08/19 22:12:56 d2.data.common]: Serializing the dataset using: <class 'detectron2.data.common._TorchSerializedList'>
[08/19 22:12:56 d2.data.common]: Serializing 2453 elements to byte tensors and concatenating them all ...
[08/19 22:12:57 d2.data.common]: Serialized dataset takes 87.84 MiB
[08/19 22:12:57 d2.data.build]: Making 

2024-08-19 22:13:28.587269: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2024-08-19 22:13:28.587328: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2024-08-19 22:13:28.588935: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


[08/19 22:13:56 d2.utils.events]:  eta: 9:25:26  iter: 30039  total_loss: 90.65  loss_ce: 0.663  loss_mask: 0.3043  loss_dice: 3.054  loss_bbox: 1.083  loss_giou: 2.242  loss_ce_dn: 0  loss_mask_dn: 0  loss_dice_dn: 0  loss_bbox_dn: 0  loss_giou_dn: 0  loss_ce_0: 0.9738  loss_mask_0: 0.3155  loss_dice_0: 3.333  loss_bbox_0: 1.254  loss_giou_0: 2.147  loss_ce_dn_0: 0  loss_mask_dn_0: 0  loss_dice_dn_0: 0  loss_bbox_dn_0: 0  loss_giou_dn_0: 0  loss_ce_1: 0.8268  loss_mask_1: 0.3441  loss_dice_1: 3.125  loss_bbox_1: 1.189  loss_giou_1: 2.195  loss_ce_dn_1: 0  loss_mask_dn_1: 0  loss_dice_dn_1: 0  loss_bbox_dn_1: 0  loss_giou_dn_1: 0  loss_ce_2: 0.7786  loss_mask_2: 0.3403  loss_dice_2: 3.181  loss_bbox_2: 1.168  loss_giou_2: 2.223  loss_ce_dn_2: 0  loss_mask_dn_2: 0  loss_dice_dn_2: 0  loss_bbox_dn_2: 0  loss_giou_dn_2: 0  loss_ce_3: 0.7297  loss_mask_3: 0.3586  loss_dice_3: 3.175  loss_bbox_3: 1.151  loss_giou_3: 2.236  loss_ce_dn_3: 0  loss_mask_dn_3: 0  loss_dice_dn_3: 0  loss_bbox_dn_

In [ ]:
!cp /kaggle/working/SwinDocSegmenter/model_final_publay_swindocseg.pth /kaggle/working/

In [ ]:
import os

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "max_split_size_mb:512"

In [ ]:
import torch
torch.cuda.empty_cache() 


In [ ]:
import gc
torch.cuda.empty_cache()
gc.collect()

In [ ]:
import json
import os
from PIL import Image

# Paths
json_file_path = '/kaggle/input/banlad2599-bangla-newspaper-layout-dataset/merged_coco.json'
images_folder_path = '/kaggle/input/banlad2599-bangla-newspaper-layout-dataset/All_Crumpled_Images'
new_json_file_path = 'updated_coco.json'

# Load the existing COCO JSON file
with open(json_file_path, 'r') as file:
    coco_data = json.load(file)

# Create a mapping of image filenames to their sizes
image_size_map = {}
for image_info in coco_data['images']:
    image_file_name = image_info['file_name']
    image_path = os.path.join(images_folder_path, image_file_name)
    if os.path.exists(image_path):
        with Image.open(image_path) as img:
            width, height = img.size
            image_size_map[image_file_name] = (width, height)

# Update the COCO JSON data with the correct sizes
for image_info in coco_data['images']:
    image_file_name = image_info['file_name']
    if image_file_name in image_size_map:
        width, height = image_size_map[image_file_name]
        image_info['width'] = width
        image_info['height'] = height

# Save the updated COCO JSON data to a new file
with open(new_json_file_path, 'w') as file:
    json.dump(coco_data, file, indent=4)

print(f"Updated COCO JSON file saved to {new_json_file_path}")